# 📖 第三课：模型评估与交叉验证

**目标**：学会量化模型质量，掌握从准确率到稳健验证的完整评估体系

---

## 📚 学习目标
- 理解训练/验证/测试三分法的意义
- 掌握准确率、精确率、召回率、F1分数的计算
- 学会使用混淆矩阵分析分类结果
- 理解 K 折交叉验证的原理和实现
- 了解 ROC 曲线和 AUC 指标

In [ ]:
# 导入必要的库
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

print("库导入成功！")

---

## 1. 为什么需要评估？

训练了一个模型之后，最关键的问题是：**它好不好？**

- **避免过拟合**：模型可能记住了训练数据但泛化能力差
- **模型选择**：在多个候选模型中选择最佳者
- **性能保证**：确保模型在实际应用中表现良好

### 训练/验证/测试三分法
```
训练集 → 训练模型（学习参数）
验证集 → 调参、选择模型（调超参数）
测试集 → 最终评估（只看一次）
```

In [ ]:
# 生成示例数据
np.random.seed(42)
X = np.random.randn(300, 2)
y = (X[:, 0] + X[:, 1] > 0).astype(int)

print(f"数据集大小: {len(X)} 个样本")
print(f"正类比例: {y.mean():.2%}")
print(f"负类比例: {1 - y.mean():.2%}")

# 三分法分割
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

print(f"\n数据分割:")
print(f"  训练集: {len(X_train)} 样本 ({len(X_train)/len(X):.0%})")
print(f"  验证集: {len(X_val)} 样本 ({len(X_val)/len(X):.0%})")
print(f"  测试集: {len(X_test)} 样本 ({len(X_test)/len(X):.0%})")

---

## 2. 混淆矩阵与四大指标

In [ ]:
# 训练逻辑回归模型
model = LogisticRegression()
model.fit(X_train, y_train)

# 在测试集上预测
y_pred = model.predict(X_test)

# 混淆矩阵
cm = confusion_matrix(y_test, y_pred)
print("混淆矩阵:")
print("          预测负类  预测正类")
print(f"实际负类   {cm[0][0]:5d}     {cm[0][1]:5d}")
print(f"实际正类   {cm[1][0]:5d}     {cm[1][1]:5d}")
print(f"\nTP={cm[1][1]}, FP={cm[0][1]}, FN={cm[1][0]}, TN={cm[0][0]}")

In [ ]:
# 可视化混淆矩阵
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, cmap='Blues', interpolation='nearest')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['预测负类', '预测正类'])
ax.set_yticklabels(['实际负类', '实际正类'])
ax.set_xlabel('预测标签', fontsize=12)
ax.set_ylabel('真实标签', fontsize=12)
ax.set_title('混淆矩阵', fontsize=14)

# 在每个格子上显示数字
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i][j]), ha='center', va='center', fontsize=20,
                color='white' if cm[i][j] > cm.max()/2 else 'black')
plt.colorbar(im)
plt.tight_layout()
plt.show()

In [ ]:
# 计算四大核心指标
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("="*50)
print("四大核心评估指标")
print("="*50)
print(f"准确率 (Accuracy):  {accuracy:.3f}  — 所有预测中正确的比例")
print(f"精确率 (Precision): {precision:.3f}  — 预测为正的中有多少是真正")
print(f"召回率 (Recall):    {recall:.3f}  — 真正中有多少被找出来")
print(f"F1 分数:            {f1:.3f}  — 精确率和召回率的调和平均")
print("="*50)

# 使用 classification_report 一次性输出
print("\n完整分类报告:")
print(classification_report(y_test, y_pred, target_names=['负类', '正类']))

---

## 3. 精确率 vs 召回率的权衡

In [ ]:
# 不同阈值下精确率和召回率的变化
y_proba = model.predict_proba(X_test)[:, 1]  # 正类概率

thresholds = np.arange(0.1, 1.0, 0.1)
precisions = []
recalls = []
f1s = []

print("阈值\t精确率\t召回率\tF1")
print("-" * 35)
for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    p = precision_score(y_test, y_pred_t, zero_division=0)
    r = recall_score(y_test, y_pred_t, zero_division=0)
    f = f1_score(y_test, y_pred_t, zero_division=0)
    precisions.append(p)
    recalls.append(r)
    f1s.append(f)
    print(f"{t:.1f}\t{p:.3f}\t{r:.3f}\t{f:.3f}")

# 可视化
plt.figure(figsize=(10, 6))
plt.plot(thresholds, precisions, 'b-o', label='精确率', markersize=6)
plt.plot(thresholds, recalls, 'r-s', label='召回率', markersize=6)
plt.plot(thresholds, f1s, 'g-^', label='F1分数', markersize=6)
plt.xlabel('决策阈值')
plt.ylabel('指标值')
plt.title('不同阈值下的指标变化')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---

## 4. K 折交叉验证

In [ ]:
# 5折交叉验证
model = LogisticRegression()

# 使用不同指标进行交叉验证
scoring = ['accuracy', 'precision', 'recall', 'f1']

print("5折交叉验证结果:")
print("="*50)
for metric in scoring:
    scores = cross_val_score(model, X, y, cv=5, scoring=metric)
    print(f"{metric:12s}: {scores.mean():.3f} ± {scores.std():.3f}")
    print(f"             各折分数: {scores.round(3)}")
print("="*50)
print("\n💡 交叉验证的优势:")
print("  - 每个样本都会被用作训练和测试")
print("  - 结果更稳定，不受单次分割的影响")
print("  - 标准差反映模型稳定性")

---

## 5. ROC 曲线与 AUC

In [ ]:
# 计算ROC曲线
y_proba = model.fit(X_train, y_train).predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc_score = roc_auc_score(y_test, y_proba)

# 可视化ROC曲线
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'逻辑回归 (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], 'r--', linewidth=1, label='随机分类器 (AUC = 0.500)')
plt.fill_between(fpr, tpr, alpha=0.1, color='blue')
plt.xlabel('假正率 (False Positive Rate)', fontsize=12)
plt.ylabel('真正率 (True Positive Rate)', fontsize=12)
plt.title('ROC 曲线', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print(f"AUC 分数: {auc_score:.3f}")
print("\n💡 AUC 的含义:")
print("  - 1.0 = 完美分类器")
print("  - 0.5 = 随机猜测")
print("  - AUC 越大，模型区分正负类的能力越强")

---

## 6. 综合实战：对比不同模型

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# 准备多个候选模型
models = {
    '逻辑回归': LogisticRegression(),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    '决策树 (depth=3)': DecisionTreeClassifier(max_depth=3, random_state=42)
}

print("模型对比 (5折交叉验证)")
print("="*60)
print(f"{'模型':20s} {'准确率':>10s} {'精确率':>10s} {'召回率':>10s} {'F1':>10s}")
print("-" * 60)

for name, model in models.items():
    acc = cross_val_score(model, X, y, cv=5, scoring='accuracy').mean()
    prec = cross_val_score(model, X, y, cv=5, scoring='precision').mean()
    rec = cross_val_score(model, X, y, cv=5, scoring='recall').mean()
    f1 = cross_val_score(model, X, y, cv=5, scoring='f1').mean()
    print(f"{name:20s} {acc:10.3f} {prec:10.3f} {rec:10.3f} {f1:10.3f}")

print("="*60)

---

## 📝 课后思考

1. 在疾病检测场景中，精确率和召回率哪个更重要？为什么？
2. 如果数据极度不平衡（正类只有1%），准确率还有意义吗？
3. 交叉验证的 K 值选多少合适？K 越大越好吗？

---

**下一步**：过拟合与正则化——学习如何处理模型在训练集上表现好但测试集差的经典问题。